In [25]:
#Extracting data from a exhibition webpage. Information of exhibitor,industry,  country, email, location, stall, and any phone number if available. 
#There are 1500 plus exhibitors

In [32]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://sahaexpo.com"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [33]:
# url = f"{BASE_URL}/en/exhibitors?page=1"
    
# res = requests.get(url, headers=HEADERS)
# soup = BeautifulSoup(res.text, "html.parser")
    
# #links = []
    
# cards = soup.select("a.exhibitor-card")
# print(cards[0])
# href = card.get("href")

# if href.startswith("http"):
#     full_url = href
# else:
#     full_url = BASE_URL + href

In [34]:
# -------------------------------
# STEP 1: Get all exhibitor links from 1 page
# -------------------------------
def get_exhibitor_links(page):
    url = f"{BASE_URL}/en/exhibitors?page={page}"
    
    res = requests.get(url, headers=HEADERS)
    res.encoding = "utf-8"  
    soup = BeautifulSoup(res.text, "html.parser")
    
    links = []
    
    cards = soup.select("a.exhibitor-card")
    
    for card in cards:
        href = card.get("href")
        if href.startswith("http"):
            full_url=href
        else:
            full_url=BASE_URL + href
        
        links.append(full_url)
    
    return links

In [35]:
def decode_cf_email(cfemail):
    r = int(cfemail[:2], 16)
    email = ''.join(
        chr(int(cfemail[i:i+2], 16) ^ r)
        for i in range(2, len(cfemail), 2)
    )
    return email

# STEP 2: Extract data from detail page
# -------------------------------
def parse_exhibitor(url):
    res = requests.get(url, headers=HEADERS)
    res.encoding = "utf-8"  
    soup = BeautifulSoup(res.text, "html.parser")
    
    data = {}

    # --- Name
    name = soup.select_one("h1")
    data["name"] = name.text.strip() if name else None

    # --- Hall & Stand
    location = soup.select_one(".location-text")

    if location:
        hall = location.find("strong", string="Hall:")
        stand = location.find("strong", string="Stand:")
        
        data["hall"] = hall.next_sibling.strip() if hall else None
        data["stand"] = stand.next_sibling.strip() if stand else None
    else:
        data["hall"] = None
        data["stand"] = None

    # --- Industry (Business Areas)
    industries = soup.select(".exhibitor-category-tag-sm")
    industry_list = [i.get_text(strip=True) for i in industries]
    data["industry"] = ", ".join(industry_list) if industry_list else None

    # --- Contact Info
    # contact_items = soup.select(".exhibitor-contact-item span")
    # texts = [item.get_text(strip=True) for item in contact_items]

    # data["website"] = None
    # data["email"] = None
    # data["address"] = None
    # data["country"] = None

    # if len(texts) == 4:
    #     data["email"] = texts[0]
    #     data["website"] = texts[1]
    #     data["address"] = texts[2]
    #     data["country"] = texts[3]

    # elif len(texts) == 3:
    #     data["email"] = texts[0]
    #     data["website"] = texts[1]
    #     data["country"] = texts[2]
    contact_items = soup.select(".exhibitor-contact-item")
    texts = [item.get_text(strip=True) for item in contact_items]

# Email (special handling)
    email_tag = soup.select_one(".__cf_email__")
    if email_tag:
        data["email"] = decode_cf_email(email_tag.get("data-cfemail"))
        
    
    else:
        data["email"] = None
   

# Other fields
    data["website"] = None
    data["address"] = None
    data["country"] = None

    if len(texts) == 4:
        data["website"] = texts[1]
        data["address"] = texts[2]
        data["country"] = texts[3]
        data["country"] = data["country"].split()[-1] if data["country"] else None
    

    elif len(texts) == 3:
        data["website"] = texts[1]
        data["country"] = texts[2]
        data["country"] = data["country"].split()[-1] if data["country"] else None
    
        
    
        
    

   

    # --- About
    about = soup.select_one(".exhibitor-about-content")
    data["about"] = about.get_text(" ", strip=True) if about else None

    # --- URL
    data["url"] = url

    return data


In [36]:
links = get_exhibitor_links(page=1)

print("Total links:", len(links))
print(links[:3])  # just to verify

Total links: 24
['https://sahaexpo.com/en/katilimci/2j-antennas', 'https://sahaexpo.com/en/katilimci/3d-systems-inc-corporation', 'https://sahaexpo.com/en/katilimci/3dlab-arcway']


In [37]:
for link in links[:3]:
    data = parse_exhibitor(link)
    print(data)
    print("-" * 50)

{'name': '2J Antennas', 'hall': '2', 'stand': '2C-05', 'industry': None, 'email': 'info@e-kom.com', 'website': 'www.2j-antennas.com', 'address': 'Stefanikova 61 085 01 Bardejov Slovakia', 'country': 'Slovakia', 'about': '2J Antennas’ vision is to be a global, world-class trusted leader in antenna solutions that aims to deliver advanced technologies for the connected world. We innovate and offer high-quality products and services with cutting-edge technologies to meet the rapidly evolving wireless industry catering to 5G, 4G (LTE), 3G, 2G, WiFi, Iridium communications, GPS/GNSS/BeiDou frequencies and more for the automotive, navigation, IoT, marine, telematic, automation, M2M and other markets.', 'url': 'https://sahaexpo.com/en/katilimci/2j-antennas'}
--------------------------------------------------
{'name': '3D Systems, Inc. Corporation', 'hall': '7', 'stand': '7A-15', 'industry': None, 'email': 'moreinfo@3dsystems.com', 'website': 'www.3dsystems.com', 'address': '3D Systems Corporat

In [38]:
all_data = []

for link in links:
    try:
        data = parse_exhibitor(link)
        all_data.append(data)
        time.sleep(1)   # IMPORTANT (avoid blocking)
    except Exception as e:
        print("Error:", link, e)

In [39]:
df = pd.DataFrame(all_data)

In [40]:
df.head()

,name,hall,stand,industry,email,website,address,country,about,url
0,2J Antennas,2,2C-05,None,info@e-kom.com,www.2j-antennas.com,Stefanikova 61 085 01 Bardejov Slovakia,Slovakia,"2J Antennas’ vision is to be a global, world-c...",https://sahaexpo.com/en/katilimci/2j-antennas
1,"3D Systems, Inc. Corporation",7,7A-15,None,moreinfo@3dsystems.com,www.3dsystems.com,3D Systems Corporation 333 Three D Systems Cir...,States,"More than 35 years ago, Chuck Hull’s curiosity...",https://sahaexpo.com/en/katilimci/3d-systems-i...
2,3DLAB (ARCWAY),3,3E-09,None,info@voksel.com.tr,www.voksel.com.tr,Turgut Özal Mh. Tonguç Baba Cd. 21/11 Esenyurt...,Türkiye,None,https://sahaexpo.com/en/katilimci/3dlab-arcway
3,3DWOVENS,6,6F-01b,"Composite Material Technologies, Aircraft and ...",info@3dwovens.com,www.3dwovens.com,Yenibosna Merkez Mah. Kor Sk. No:9 Bahçelievle...,Türkiye,3DWovens: Advanced 3D Composite Solutions for ...,https://sahaexpo.com/en/katilimci/3dwovens
4,3EEOS,4,4F-13,Optical Material and Electro-optic Device Tech...,support@3eeos.com,www.3eeos.com,"ASO 1.Org. San. Bölg. Anadolu Cad. No:, D:5, 0...",Türkiye,"For over 20 years, 3E has stood at the interse...",https://sahaexpo.com/en/katilimci/3eeos


In [41]:
df.to_csv("saha_page1.csv", index=False)

In [43]:
all_links = []

for page in range(1, 62):   # ~60 pages
    print(f"Scraping page {page}")
    links = get_exhibitor_links(page)
    all_links.extend(links)
    time.sleep(1)

Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5
Scraping page 6
Scraping page 7
Scraping page 8
Scraping page 9
Scraping page 10
Scraping page 11
Scraping page 12
Scraping page 13
Scraping page 14
Scraping page 15
Scraping page 16
Scraping page 17
Scraping page 18
Scraping page 19
Scraping page 20
Scraping page 21
Scraping page 22
Scraping page 23
Scraping page 24
Scraping page 25
Scraping page 26
Scraping page 27
Scraping page 28
Scraping page 29
Scraping page 30
Scraping page 31
Scraping page 32
Scraping page 33
Scraping page 34
Scraping page 35
Scraping page 36
Scraping page 37
Scraping page 38
Scraping page 39
Scraping page 40
Scraping page 41
Scraping page 42
Scraping page 43
Scraping page 44
Scraping page 45
Scraping page 46
Scraping page 47
Scraping page 48
Scraping page 49
Scraping page 50
Scraping page 51
Scraping page 52
Scraping page 53
Scraping page 54
Scraping page 55
Scraping page 56
Scraping page 57
Scraping page 58
Scraping page 59
Scrapi

In [ ]:
all_data = []

for i, link in enumerate(all_links):
    print(f"{i+1}/{len(all_links)}")
    
    try:
        data = parse_exhibitor(link)
        all_data.append(data)
        time.sleep(1)
    except Exception as e:
        print("Error:", link)

1/1457
2/1457
3/1457
4/1457
5/1457
6/1457
7/1457
8/1457
9/1457
10/1457
11/1457
12/1457
13/1457
14/1457
15/1457
16/1457
17/1457
18/1457
19/1457
20/1457
21/1457
22/1457
23/1457
24/1457
25/1457
26/1457
27/1457
28/1457
29/1457
30/1457
31/1457
32/1457
33/1457
34/1457
35/1457
36/1457
37/1457
38/1457
39/1457
40/1457
41/1457
42/1457
43/1457
44/1457
45/1457
46/1457
47/1457
48/1457
49/1457
50/1457
51/1457
52/1457
53/1457
54/1457
55/1457
56/1457
57/1457
58/1457
59/1457
60/1457
61/1457
62/1457
63/1457
64/1457
65/1457
66/1457
67/1457
68/1457
69/1457
70/1457
71/1457
72/1457
73/1457
74/1457
75/1457
76/1457
77/1457
78/1457
79/1457
80/1457
81/1457
82/1457
83/1457
84/1457
85/1457
86/1457
87/1457
88/1457
89/1457
90/1457
91/1457
92/1457
93/1457
94/1457
95/1457
96/1457
97/1457
98/1457
99/1457
100/1457
101/1457
102/1457
103/1457
104/1457
105/1457
106/1457
107/1457
108/1457
109/1457
110/1457
111/1457
112/1457
113/1457
114/1457
115/1457
116/1457
117/1457
118/1457
119/1457
120/1457
121/1457
122/1457
123/1457
1

In [45]:
df = pd.DataFrame(all_data)
df.to_csv("saha_full_data.csv", index=False)